# Bước 14 – Ba loại mô hình: LightGBM, MLP, cây quyết định

Kiểm tra xem 3 kết luận chính có còn đúng khi đổi loại mô hình không:
1. không có nhãn thì chuyên gia từ mạng khác gần như đoán mò;
2. vài trăm nhãn thật của mạng mới đã đủ tốt (M2);
3. ngưỡng tự điều chỉnh theo cửa sổ thời gian (chống sập 10%) giúp ích.

Lý thuyết (kèm ví dụ số) nằm ở Phần 6 của tài liệu "TADP-IDS – Lý thuyết bước 13".

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Chạy 3 loại mô hình × 4 mạng | 40–90 phút |

**Nên dùng máy KHÔNG có GPU (card đồ họa)** (Runtime → Change runtime type → CPU): MLP chạy trên GPU cho kết quả hơi khác nhau giữa các lần chạy; trên CPU thì luôn giống nhau. Kết quả lưu vào `MyDrive/TADP_Q1/results/b14_ba_mo_hinh/`. Nếu Colab ngắt, chạy lại Ô 2: phần nào xong rồi sẽ được bỏ qua.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
import torch
print("GPU:", "CÓ – nên đổi sang CPU" if torch.cuda.is_available() else "không có (đúng như mong muốn)")
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5934,
            "few_label.py": 15932, "metrics.py": 4002, "model_families.py": 10460, "preprocess.py": 3601,
            "shift.py": 3863, "window_threshold.py": 12240}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 14" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
GPU: không có (đúng như mong muốn)
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 14


## Ô 2 – Chạy

Với mỗi loại mô hình: 4 dòng "chuyên gia … xong", rồi 4 dòng "[tên mạng] xong". Cuối cùng in:
- Bảng 14.1: Macro-F1 trung bình 4 mạng, cột = (loại mô hình, cách đặt ngưỡng a / e / h10);
- Bảng 14.2: AUROC trung bình 4 mạng;
- Bảng 14.3: Macro-F1 từng mạng (ngưỡng h10);
- Bảng 14.4: số lần M2 không học được vì k nhãn có ít hơn 3 luồng tấn công (vẫn được tính vào trung bình như một lần thất bại).

Chạy xong báo Claude.

In [2]:
import sys, warnings
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.model_families import run_families
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b14_ba_mo_hinh",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_families(cfg, data, splits, cols, pv="P1_log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
[lgbm] chuyên gia CIC-IDS2017     : xong (15 giây)
[lgbm] chuyên gia CSE-CIC-IDS2018 : xong (15 giây)
[lgbm] chuyên gia CIC-ToN-IoT     : xong (14 giây)
[lgbm] chuyên gia CIC-UNSW-NB15   : xong (12 giây)
[lgbm] [CIC-IDS2017] xong trong 119 giây
[lgbm] [CSE-CIC-IDS2018] xong trong 115 giây
[lgbm] [CIC-ToN-IoT] xong trong 113 giây
[lgbm] [CIC-UNSW-NB15] xong trong 94 giây
[mlp] chuyên gia CIC-IDS2017     : xong (22 giây)
[mlp] chuyên gia CSE-CIC-IDS2018 : xong (14 giây)
[mlp] chuyên gia CIC-ToN-IoT     : xong (17 giây)
[mlp] chuyên gia CIC-UNSW-NB15   : xong (10 giây)
[mlp] [CIC-IDS2017] xong trong 84 giây
[mlp] [CSE-CI

mo_hinh                             lgbm                                            mlp                                           tree                                     
cach_nguong                    a_co_dinh e_otsu_cua_so_co_chan h_chong_sap_10 a_co_dinh e_otsu_cua_so_co_chan h_chong_sap_10 a_co_dinh e_otsu_cua_so_co_chan h_chong_sap_10
phuong_phap             k                                                                                                                                                  
M2                      100        0.762                 0.755          0.760     0.781                 0.745          0.768     0.728                 0.734          0.726
                        500        0.827                 0.908          0.855     0.902                 0.873          0.900     0.836                 0.848          0.828
                        1000       0.784                 0.882          0.820     0.911                 0.891          0.903     0.812                 0.799          0.783
                        5000       0.843                 0.938          0.938     0.940                 0.914          0.926     0.831                 0.832          0.821
k0_chon_kiem_dinh_nguon 0          0.492                 0.569          0.569     0.573                 0.493          0.493     0.550                 0.579          0.579
k0_trung_binh_deu       0          0.457                 0.387          0.387     0.500                 0.480          0.480     0.432                 0.321          0.321
tran_du_nhan            200000     0.970                 0.765          0.970     0.961                 0.950          0.950     0.965                 0.956          0.956